In [2]:
from datasets import load_dataset
ds1 = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
ds2 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")
updated = {}
for i in ds2:
    updated[(i['client_hash_id'], i['content_hash_id'])] = i['content_updated_date']
count = matched = 0
dic = {}
for i in ds1:
    count += 1
    if count % 10**5 == 0: print(count, matched)
    if not i["gsc_data_available"]: continue 
    t = (i['client_hash_id'], i['content_hash_id'])
    if t not in updated or updated[t] > i['report_date']: continue
    if t not in dic: dic[t] = set()
    dic[t].add(i['report_date'])
    matched += 1

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100000 0
200000 0
300000 0
400000 0
500000 0
600000 0
700000 0
800000 0
900000 0
1000000 0
1100000 1
1200000 2
1300000 98
1400000 222
1500000 336
1600000 442
1700000 490
1800000 856
1900000 1182
2000000 1495
2100000 1753
2200000 1880
2300000 2418
2400000 2644
2500000 2984
2600000 3302
2700000 3393
2800000 3530
2900000 3962
3000000 4597
3100000 4886
3200000 5158
3300000 5370
3400000 5489
3500000 5677
3600000 5762
3700000 5831
3800000 5991
3900000 6245
4000000 6326
4100000 6418
4200000 6658
4300000 6685
4400000 6780
4500000 6884
4600000 7063
4700000 7107
4800000 7131
4900000 7288
5000000 7378
5100000 7484
5200000 7584
5300000 7643
5400000 7850
5500000 7889
5600000 8553
5700000 8752
5800000 9816
5900000 10381
6000000 11613
6100000 11790
6200000 12812
6300000 14492
6400000 15563
6500000 16169
6600000 16192
6700000 16773
6800000 16995
6900000 17362
7000000 18360
7100000 18948
7200000 19421
7300000 19792
7400000 20039
7500000 20240
7600000 21146
7700000 21922
7800000 22125
7900000 23308
8000

In [3]:
from datetime import timedelta
import random
random.seed(42)
def get_streaks(dates):
    if len(dates) == 0: return []
    dates = sorted(dates)
    streaks = [[dates[0]]]
    for i in range(1, len(dates)):
        if dates[i] == dates[i-1] + timedelta(days=1):
            streaks[-1].append(dates[i])
            if len(streaks[-1]) == 90:
                streaks.append(streaks[-1][15:])
                streaks[-2] = (streaks[-2][0], streaks[-2][-1])
        else: 
            streaks[-1] = [dates[i]]
    if type(streaks[-1]) == list: streaks.pop()
    return streaks
streaks = []
for i in dic:
    s = get_streaks(dic[i])
    if len(s) > 0: streaks.append((i, s))
total = sum(len(streak) for _, streak in streaks)
random.shuffle(streaks)
valindex = testindex = 0
cur = 0
for i in range(len(streaks)):
    cur += len(streaks[i][1])
    if cur >= 0.8 * total and valindex == 0: valindex = i+1
    if cur >= 0.9 * total and testindex == 0: testindex = i+1
train_pages = streaks[:valindex]
val_pages = streaks[valindex:testindex]
test_pages = streaks[testindex:]
page_blocks = {}
data_id = 0
splits = {"train": train_pages, "val": val_pages, "test": test_pages}
for split_name, pages in splits.items():
    for page, blocks in pages:
        page_blocks[page] = {"split": split_name, "blocks": []}
        for start, end in blocks:
            page_blocks[page]["blocks"].append((start, end, data_id))
            data_id += 1

In [4]:
content_info = {}
for i in ds2:
    content_info[(i['client_hash_id'], i['content_hash_id'])] = {'word_count': i['word_count'],
                                                                'update_date': i['content_updated_date'], 
                                                                'create_date': i['content_created_date'], 
                                                                'content_type': i['content_type'],
                                                                'main_intent': i['main_intent']}
data = {'train': {}, 'val': {}, 'test': {}}
for _, i in page_blocks.items():
    for start, end, data_id in i['blocks']:
        data[i['split']][data_id] = {'clicks': [0] * 5, 
                                     'impressions': [0] * 5, 
                                     'position': [0] * 5,
                                     'word_count': None,
                                     'update_date': None,
                                     'create_date': None,
                                     'content_type': None,
                                     'main_intent': None}
count = matched = 0
for i in ds1:
    t = (i['client_hash_id'], i['content_hash_id'])
    count += 1
    if count % 10**5 == 0: print(count, matched)
    if t not in page_blocks: continue
    for start, end, data_id in page_blocks[t]['blocks']:
        if start <= i['report_date'] <= end:
            delta = (i['report_date'] - start).days
            data[page_blocks[t]['split']][data_id]['clicks'][min(4, delta // 15)] += i['gsc_clicks']
            data[page_blocks[t]['split']][data_id]['impressions'][min(4, delta // 15)] += i['gsc_impressions']
            data[page_blocks[t]['split']][data_id]['position'][min(4, delta // 15)] += i['gsc_sum_position']
            data[page_blocks[t]['split']][data_id]['word_count'] = content_info[t]['word_count']
            data[page_blocks[t]['split']][data_id]['update_date'] = (start - content_info[t]['update_date']).days
            data[page_blocks[t]['split']][data_id]['create_date'] = (start - content_info[t]['create_date']).days
            data[page_blocks[t]['split']][data_id]['content_type'] = content_info[t]['content_type']
            data[page_blocks[t]['split']][data_id]['main_intent'] = content_info[t]['main_intent']
            matched += 1
for split in data:
    for i in data[split]:
        for j in range(5):
            if data[split][i]['impressions'][j] == 0: data[split][i]['position'][j] = None
            else: data[split][i]['position'][j] /= data[split][i]['impressions'][j]

100000 0
200000 0
300000 0
400000 0
500000 0
600000 0
700000 0
800000 0
900000 0
1000000 0
1100000 0
1200000 0
1300000 3
1400000 9
1500000 14
1600000 19
1700000 23
1800000 92
1900000 145
2000000 212
2100000 253
2200000 267
2300000 379
2400000 424
2500000 620
2600000 831
2700000 883
2800000 947
2900000 1173
3000000 1475
3100000 1607
3200000 1731
3300000 1942
3400000 2029
3500000 2224
3600000 2294
3700000 2361
3800000 2508
3900000 2785
4000000 2871
4100000 2955
4200000 3304
4300000 3304
4400000 3395
4500000 3486
4600000 3685
4700000 3685
4800000 3685
4900000 3888
5000000 3989
5100000 4094
5200000 4186
5300000 4186
5400000 4499
5500000 4499
5600000 4607
5700000 4826
5800000 4863
5900000 4863
6000000 5011
6100000 5018
6200000 5035
6300000 5108
6400000 5153
6500000 5153
6600000 5153
6700000 5263
6800000 5596
6900000 5596
7000000 5630
7100000 5630
7200000 5630
7300000 5747
7400000 5854
7500000 5865
7600000 5900
7700000 6126
7800000 6344
7900000 6391
8000000 6468
8100000 6468
8200000 6481
830

In [5]:
import numpy as np
X_train, y_train1, y_train2, y_train3 = [], [], [], []
main_intent = {'informational': 0, 'commercial': 1, 'transactional': 2, 'navigational': 3, None: None}
for i in data['train'].values():
    X_train.append(i['clicks'][:4] 
                   + i['impressions'][:4] 
                   + i['position'][:4] 
                   + [i['word_count']]
                   + [i['update_date']] 
                   + [i['create_date']] 
                   + [main_intent[i['main_intent']]])
    y_train1.append(i['clicks'][4])
    y_train2.append(i['impressions'][4])
    y_train3.append(i['position'][4])
X_train, y_train1, y_train2, y_train3 = np.array(X_train), np.array(y_train1), np.array(y_train2), np.array(y_train3)
X_val, y_val1, y_val2, y_val3 = [], [], [], []
for i in data['val'].values():
    X_val.append(i['clicks'][:4] 
                + i['impressions'][:4] 
                + i['position'][:4] 
                + [i['word_count']] 
                + [i['update_date']] 
                + [i['create_date']] 
                + [main_intent[i['main_intent']]])
    y_val1.append(i['clicks'][4])
    y_val2.append(i['impressions'][4])
    y_val3.append(i['position'][4])
X_val, y_val1, y_val2, y_val3 = np.array(X_val), np.array(y_val1), np.array(y_val2), np.array(y_val3)
X_test, y_test1, y_test2, y_test3 = [], [], [], []
for i in data['test'].values():
    X_test.append(i['clicks'][:4] 
                + i['impressions'][:4] 
                + i['position'][:4] 
                + [i['word_count']] 
                + [i['update_date']] 
                + [i['create_date']] 
                + [main_intent[i['main_intent']]])
    y_test1.append(i['clicks'][4])
    y_test2.append(i['impressions'][4])
    y_test3.append(i['position'][4])
X_test, y_test1, y_test2, y_test3 = np.array(X_test), np.array(y_test1), np.array(y_test2), np.array(y_test3)

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
def test_model(X_train, y_train, X_val, y_val):
    best = [(0, 0, 0, 0), float('-inf')]
    for max_iter in [100, 200, 300, 500, 800]:
        for learning_rate in [0.01, 0.025, 0.05, 0.1]:
            for min_samples_leaf in [10, 20, 30, 50]:
                for l2_regularization in [0, 0.1, 1, 10]:
                    model = HistGradientBoostingRegressor(
                            max_depth=5,
                            max_iter=max_iter,
                            learning_rate=learning_rate,
                            min_samples_leaf=min_samples_leaf,
                            l2_regularization=l2_regularization,
                            early_stopping=False,
                            random_state=42)
                    model.fit(X_train, y_train)
                    pred = model.predict(X_val)
                    r2 = r2_score(y_val, pred)
                    if r2 > best[1]: best = [(max_iter, learning_rate, min_samples_leaf, l2_regularization), r2]
    return best
best1 = test_model(X_train, y_train1, X_val, y_val1)
best2 = test_model(X_train, y_train2, X_val, y_val2)
best3 = test_model(X_train, y_train3, X_val, y_val3)
model1 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best1[0][0],
    learning_rate=best1[0][1],
    min_samples_leaf=best1[0][2],
    l2_regularization=best1[0][3],
    early_stopping=False,
    random_state=42
)
model2 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best2[0][0],
    learning_rate=best2[0][1],
    min_samples_leaf=best2[0][2],
    l2_regularization=best2[0][3],
    early_stopping=False,
    random_state=42
)
model3 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best3[0][0],
    learning_rate=best3[0][1],
    min_samples_leaf=best3[0][2],
    l2_regularization=best3[0][3],
    early_stopping=False,
    random_state=42
)
model1.fit(X_train, y_train1)
pred1 = model1.predict(X_val)
print(f'Clicks: r2 score = {best1[1]}, mean absolute error = {mean_absolute_error(y_val1, pred1)}')
model2.fit(X_train, y_train2)
pred2 = model2.predict(X_val)
print(f'Impressions: r2 score = {best2[1]}, mean absolute error = {mean_absolute_error(y_val2, pred2)}')
model3.fit(X_train, y_train3)
pred3 = model3.predict(X_val)
print(f'Position: r2 score = {best3[1]}, mean absolute error = {mean_absolute_error(y_val3, pred3)}')

Clicks: r2 score = 0.8835630020877395, mean absolute error = 2.5040178450534047
Impressions: r2 score = 0.8125844352079867, mean absolute error = 686.9619234343456
Position: r2 score = 0.8490259057310643, mean absolute error = 3.451586686627926


In [7]:
import pandas as pd
table = pd.DataFrame({
"Hyperparameter": ["max_iter", "learning_rate", "min_samples_leaf", "l2_regularization"],
    "Clicks model": [model1.max_iter, model1.learning_rate, model1.min_samples_leaf, model1.l2_regularization],
    "Impressions model": [model2.max_iter, model2.learning_rate, model2.min_samples_leaf, model2.l2_regularization],
    "Position model": [model3.max_iter, model3.learning_rate, model3.min_samples_leaf, model3.l2_regularization],
})
table.head()

,Hyperparameter,Clicks model,Impressions model,Position model
0,max_iter,100.00,200.00,500.00
1,learning_rate,0.05,0.01,0.05
2,min_samples_leaf,10.00,50.00,30.00
3,l2_regularization,0.10,10.00,1.00


In [8]:
recent_baseline1 = X_test[:, 3] * 2
print("Clicks recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test1, recent_baseline1))
print("R2 score:", r2_score(y_test1, recent_baseline1))
mean_60d_baseline1 = np.mean(X_test[:, :4], axis=1) * 2
print("\nClicks mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test1, mean_60d_baseline1))
print("R2 score:", r2_score(y_test1, mean_60d_baseline1))
model_pred1 = model1.predict(X_test)
print("\nClicks model performance:")
print("Mean absolute error:", mean_absolute_error(y_test1, model_pred1))
print("R2 score:", r2_score(y_test1, model_pred1))
recent_baseline2 = X_test[:, 7] * 2
print("\nImpressions recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test2, recent_baseline2))
print("R2 score:", r2_score(y_test2, recent_baseline2))
mean_60d_baseline2 = np.mean(X_test[:, 4:8], axis=1) * 2
print("\nImpressions mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test2, mean_60d_baseline2))
print("R2 score:", r2_score(y_test2, mean_60d_baseline2))
model_pred2 = model2.predict(X_test)
print("\nImpressions model performance:")
print("Mean absolute error:", mean_absolute_error(y_test2, model_pred2))
print("R2 score:", r2_score(y_test2, model_pred2))
recent_baseline3 = X_test[:, 11]
print("\nPosition recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test3, recent_baseline3))
print("R2 score:", r2_score(y_test3, recent_baseline3))
mean_60d_baseline3 = (np.sum(X_test[:, 4:8] * X_test[:, 8:12], axis=1) / np.sum(X_test[:, 4:8], axis=1))
print("\nPosition mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test3, mean_60d_baseline3))
print("R2 score:", r2_score(y_test3, mean_60d_baseline3))
model_pred3 = model3.predict(X_test)
print("\nPosition model performance:")
print("Mean absolute error:", mean_absolute_error(y_test3, model_pred3))
print("R2 score:", r2_score(y_test3, model_pred3))

Clicks recent 15 day baseline:
Mean absolute error: 2.8983957219251337
R2 score: 0.8162570699245038

Clicks mean 60 day baseline:
Mean absolute error: 2.825668449197861
R2 score: 0.7866514920037863

Clicks model performance:
Mean absolute error: 2.4021781561785294
R2 score: 0.8863866603138656

Impressions recent 15 day baseline:
Mean absolute error: 743.9522281639929
R2 score: 0.7633164192420292

Impressions mean 60 day baseline:
Mean absolute error: 1090.2247771836007
R2 score: 0.7057075017721606

Impressions model performance:
Mean absolute error: 603.9551392278299
R2 score: 0.8734313180224106

Position recent 15 day baseline:
Mean absolute error: 4.92981701679662
R2 score: 0.7336817334283864

Position mean 60 day baseline:
Mean absolute error: 6.073677285624553
R2 score: 0.6226871089824088

Position model performance:
Mean absolute error: 3.759330180284617
R2 score: 0.8483991611488413


In [10]:
import csv
pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)
pred3 = model3.predict(X_test)
headers = ['clicks_1_15', 'clicks_16_30', 'clicks_31_45', 'clicks_46_60', 
           'impressions_1_15', 'impressions_16_30', 'impressions_31_45', 'impressions_46_60',
           'position_1_15', 'position_16_30', 'position_31_45', 'position_46_60', 
           'word_count', 'update_date', 'create_date', 'main_intent',
           'pred_clicks', 'pred_impressions', 'pred_position',
           'true_clicks', 'true_impressions', 'true_position',
           'clicks_decline', 'impressions_decline', 'position_worsen',
           'clicks_decline_baseline', 'impressions_decline_baseline', 'position_worsen_baseline']
data = []
click_model = impression_model = position_model = click_baseline = impression_baseline = position_baseline = 0
click_model_correct = impression_model_correct = position_model_correct = click_baseline_correct = impression_baseline_correct = position_baseline_correct = 0
for i in range(len(X_test)):
    arr = X_test[i].tolist()
    arr.extend([pred1[i], pred2[i], pred3[i]])
    arr.extend([y_test1[i], y_test2[i], y_test3[i]])
    arr.append(int(pred1[i] <= 0.85 * (X_test[i][2] + X_test[i][3])))
    arr.append(int(pred2[i] <= 0.85 * (X_test[i][6] + X_test[i][7])))
    pos1 = (X_test[i][8] * X_test[i][4] + X_test[i][9] * X_test[i][5]) / (X_test[i][4] + X_test[i][5])
    pos2 = (X_test[i][10] * X_test[i][6] + X_test[i][11] * X_test[i][7]) / (X_test[i][6] + X_test[i][7])
    arr.append(int(pred3[i] >= 1.15 * pos2))
    arr.append(int(X_test[i][2] + X_test[i][3] <= 0.85 * (X_test[i][0] + X_test[i][1])))
    arr.append(int(X_test[i][6] + X_test[i][7] <= 0.85 * (X_test[i][4] + X_test[i][5])))
    arr.append(int(pos2 >= 1.15 * pos1))
    if arr[22]:
        click_model += 1
        if arr[19] <= 0.9 * (X_test[i][2] + X_test[i][3]): click_model_correct += 1
    if arr[23]:
        impression_model += 1
        if arr[20] <= 0.9 * (X_test[i][4] + X_test[i][5]): impression_model_correct += 1
    if arr[24]:
        position_model += 1
        if arr[21] >= 1.1 * pos2: position_model_correct += 1
    if arr[25]:
        click_baseline += 1
        if arr[19] <= 0.9 * (X_test[i][2] + X_test[i][3]): click_baseline_correct += 1
    if arr[26]:
        impression_baseline += 1
        if arr[20] <= 0.9 * (X_test[i][4] + X_test[i][5]): impression_baseline_correct += 1
    if arr[27]:
        position_baseline += 1
        if arr[21] >= 1.1 * pos2: position_baseline_correct += 1
    data.append(arr)
with open('work/notebooks/work/outputs.csv', mode='w', newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(data)
print(f'''Model flag precision:\nClicks: {click_model_correct / click_model} 
Impressions: {impression_model_correct / impression_model} 
Position: {position_model_correct / position_model}\n''')
print(f'''Baseline flag precision:\nClicks: {click_baseline_correct / click_baseline},
Impressions: {impression_baseline_correct / impression_baseline}, 
Position: {position_baseline_correct / position_baseline}\n''')
print(f'''Model flags:\nClicks {click_model}, Impressions: {impression_model}, Position: {position_model}\n''')
print(f'''Baseline flags:\nClicks {click_baseline}, Impressions: {impression_baseline}, Position: {position_baseline}\n''')

Model flag precision:
Clicks: 0.7053763440860215 
Impressions: 0.8279883381924198 
Position: 0.7881987577639752

Baseline flag precision:
Clicks: 0.48265368228849664,
Impressions: 0.9251533742331288, 
Position: 0.5746753246753247

Model flags:
Clicks 465, Impressions: 1029, Position: 1610

Baseline flags:
Clicks 1643, Impressions: 1630, Position: 1540



In [11]:
models = {"Clicks": (model1, y_test1), "Impressions": (model2, y_test2), "Position": (model3, y_test3)}
for name, (model, y) in models.items():
    y = np.asarray(y, dtype=float)
    pred = np.asarray(model.predict(X_test), dtype=float)
    q25, q50, q75 = np.percentile(y, [25, 50, 75])
    ranges = [("Bottom 25%", y <= q25), ("25-50%", (y > q25) & (y <= q50)), ("50-75%", (y > q50) & (y <= q75)), ("Top 25%", y > q75)]
    print(name)
    for label, mask in ranges:
        actual_mean = np.mean(y[mask])
        predicted_mean = np.mean(pred[mask])
        bias = predicted_mean - actual_mean
        print(f"{label:12} | N={np.sum(mask):4d} | Actual={actual_mean:10.2f} | Predicted = {predicted_mean:10.2f} | Bias = {bias:+10.2f}")
    print()

Clicks
Bottom 25%   | N= 842 | Actual=      0.00 | Predicted =       1.23 | Bias =      +1.23
25-50%       | N= 872 | Actual=      1.40 | Predicted =       2.11 | Bias =      +0.71
50-75%       | N= 397 | Actual=      3.41 | Predicted =       3.18 | Bias =      -0.23
Top 25%      | N= 694 | Actual=     18.18 | Predicted =      16.32 | Bias =      -1.86

Impressions
Bottom 25%   | N= 704 | Actual=    275.65 | Predicted =     675.07 | Bias =    +399.43
25-50%       | N= 700 | Actual=    523.93 | Predicted =     825.42 | Bias =    +301.49
50-75%       | N= 700 | Actual=    924.07 | Predicted =    1075.69 | Bias =    +151.62
Top 25%      | N= 701 | Actual=   5475.80 | Predicted =    4763.69 | Bias =    -712.11

Position
Bottom 25%   | N= 702 | Actual=      5.72 | Predicted =       6.69 | Bias =      +0.97
25-50%       | N= 701 | Actual=     11.87 | Predicted =      13.65 | Bias =      +1.78
50-75%       | N= 701 | Actual=     23.13 | Predicted =      24.06 | Bias =      +0.93
Top 25%      

In [12]:
recent_baseline1 = X_test[:, 3] * 2
recent_errors1 = np.abs(y_test1 - recent_baseline1)
mean_60d_baseline1 = np.mean(X_test[:, :4], axis=1) * 2
mean_60d_errors1 = np.abs(y_test1 - mean_60d_baseline1)
model_pred1 = model1.predict(X_test)
model_errors1 = np.abs(y_test1 - model_pred1)
diff1_mean_60d = mean_60d_errors1 - model_errors1
diff1_recent = recent_errors1 - model_errors1

recent_baseline2 = X_test[:, 3] * 2
recent_errors2 = np.abs(y_test2 - recent_baseline2)
mean_60d_baseline2 = np.mean(X_test[:, 4:8], axis=1) * 2
mean_60d_errors2 = np.abs(y_test2 - mean_60d_baseline2)
model_pred2 = model2.predict(X_test)
model_errors2 = np.abs(y_test2 - model_pred2)
diff2_mean_60d = mean_60d_errors2 - model_errors2
diff2_recent = recent_errors2 - model_errors2

recent_baseline3 = X_test[:, 3] * 2
recent_errors3 = np.abs(y_test3 - recent_baseline3)
mean_60d_baseline3 = np.mean(X_test[:, 8:12], axis=1) * 2
mean_60d_errors3 = np.abs(y_test3 - mean_60d_baseline3)
model_pred3 = model3.predict(X_test)
model_errors3 = np.abs(y_test3 - model_pred3)
diff3_mean_60d = mean_60d_errors3 - model_errors3
diff3_recent = recent_errors3 - model_errors3

table = pd.DataFrame([
    ["Clicks", "Mean Baseline", f"({np.mean(diff1_mean_60d) - 1.96081 * np.std(diff1_mean_60d) / 2805**0.5}, {np.mean(diff1_mean_60d) + 1.96081 * np.std(diff1_mean_60d) / 2805**0.5})"],
    ["Clicks", "Recent Baseline", f"({np.mean(diff1_recent) - 1.96081 * np.std(diff1_recent) / 2805**0.5}, {np.mean(diff1_recent) + 1.96081 * np.std(diff1_recent) / 2805**0.5})"],
    ["Impressions", "Mean Baseline", f"({np.mean(diff2_mean_60d) - 1.96081 * np.std(diff2_mean_60d) / 2805**0.5}, {np.mean(diff2_mean_60d) + 1.96081 * np.std(diff2_mean_60d) / 2805**0.5})"],
    ["Impressions", "Recent Baseline", f"({np.mean(diff2_recent) - 1.96081 * np.std(diff2_recent) / 2805**0.5}, {np.mean(diff2_recent) + 1.96081 * np.std(diff2_recent) / 2805**0.5})"],
    ["Position", "Mean Baseline", f"({np.mean(diff3_mean_60d) - 1.96081 * np.std(diff3_mean_60d) / 2805**0.5}, {np.mean(diff3_mean_60d) + 1.96081 * np.std(diff3_mean_60d) / 2805**0.5})"],
    ["Position", "Recent Baseline", f"({np.mean(diff3_recent) - 1.96081 * np.std(diff3_recent) / 2805**0.5}, {np.mean(diff3_recent) + 1.96081 * np.std(diff3_recent) / 2805**0.5})"]
], columns=["Metric", "Baseline", "Confidence interval"])
table.head(6)

,Metric,Baseline,Confidence interval
0,Clicks,Mean Baseline,"(0.2576273834940267, 0.5893532025446355)"
1,Clicks,Recent Baseline,"(0.3616090160450618, 0.6308261154481453)"
2,Impressions,Mean Baseline,"(420.5609823700921, 551.9782935414479)"
3,Impressions,Recent Baseline,"(1061.820404439257, 1316.837588049821)"
4,Position,Mean Baseline,"(10.438440729712356, 11.506756697937464)"
5,Position,Recent Baseline,"(16.627027109525883, 17.997612586045754)"
